# EE-411 — Fundamentals of Inference and Learning
## Exercise Session 11 — Diffusion models

**Learning objectives**
- Train a CNN to remove Gaussian noise from MNIST images.
- Build a forward diffusion process that gradually turns an image into noise.
- Implement deterministic and stochastic reverse diffusion and compare them.

**Prerequisites**
- Basic CNNs and Keras
- Gaussian noise and mean-squared error

Diffusion models are widely used for image, audio, and video generation. The full modern architectures can be large, but the core idea can be explored with a small model.

In this TP we use **MNIST** so that training remains reasonably light. The goal is to start from noise and generate images that look like handwritten digits.

## Part 1 — A CNN denoiser

We first train a simpler model: given a noisy image and the amount of noise, predict the clean image.


In [ ]:
from keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
import math

np.random.seed(0)

# Load MNIST and rescale pixels to [0, 1].
(x_train, _), (x_test, _) = mnist.load_data()
x_train_img = (x_train.astype("float32") / 255.0)[..., None]
x_test_img  = (x_test.astype("float32") / 255.0)[..., None]

print(x_train_img.shape)
print(x_test_img.shape)

# Plot the first ten test images.
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.imshow(x_test_img[i, ..., 0], cmap="gray")
    ax.set_axis_off()
plt.show()


### Exercise 1 — Add Gaussian noise

For a clean image $x_{\rm clean}$, we create

$$
x_{\rm noisy} = x_{\rm clean} + \sigma\,\varepsilon,
$$

where every pixel of $\varepsilon$ is sampled independently from $\mathcal N(0,1)$.

The CNN receives both the noisy image and the value of $\sigma$, and it is trained with the usual mean-squared error to reconstruct $x_{\rm clean}$.

A CNN expects image-like channels. Therefore, we copy the scalar $\sigma$ over all pixels and use it as a second channel:

- channel 1: noisy grayscale image;
- channel 2: constant image containing $\sigma$.

Complete `add_noise` below.


In [ ]:
SIGMA_SET = np.array([0.00, 0.05, 0.10, 0.20, 0.35, 0.50], dtype="float32")

def sample_sigma(batch_size):
    return np.random.choice(SIGMA_SET, size=(batch_size, 1, 1, 1)).astype("float32")

def add_noise(x, sigma):
    """
    x:     (B, 28, 28, 1) in [0, 1]
    sigma: (B, 1, 1, 1)
    """
    eps = np.random.randn(*x.shape).astype("float32")

    # TODO: add Gaussian noise with level sigma.
    # x_noisy = ...

    return np.clip(x_noisy, 0.0, 1.0)

def with_sigma_channel(x_noisy, sigma):
    """Append sigma as a constant second image channel."""
    sigma_map = np.tile(sigma, (1, 28, 28, 1))
    return np.concatenate([x_noisy, sigma_map], axis=-1)

# Sanity check.
_b = 8
_sigma = sample_sigma(_b)
_xn = add_noise(x_train_img[:_b], _sigma)
_xin = with_sigma_channel(_xn, _sigma)
print(_xin.shape)  # expected: (8, 28, 28, 2)


In [ ]:
BATCH = 128

def train_generator():
    N = x_train_img.shape[0]
    while True:
        idx = np.random.randint(0, N, size=(BATCH,))
        x_clean = x_train_img[idx]
        sigma   = sample_sigma(BATCH)
        x_noisy = add_noise(x_clean, sigma)
        x_in    = with_sigma_channel(x_noisy, sigma)
        yield x_in, x_clean

def val_generator():
    N = x_test_img.shape[0]
    while True:
        idx = np.random.randint(0, N, size=(BATCH,))
        x_clean = x_test_img[idx]
        sigma   = sample_sigma(BATCH)
        x_noisy = add_noise(x_clean, sigma)
        x_in    = with_sigma_channel(x_noisy, sigma)
        yield x_in, x_clean


### Train the denoiser

The corruption is moderate ($\sigma \leq 0.5$), so a small CNN is enough for this first experiment.


In [ ]:
from keras import layers, models, optimizers

model = models.Sequential([
    layers.Input(shape=(28, 28, 2)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.Conv2D(1, 3, padding="same", activation="sigmoid")
])

model.compile(optimizer=optimizers.Adam(2e-4), loss="mse")
model.summary()


Train the network for a few epochs.


In [ ]:
steps_per_epoch = len(x_train_img) // BATCH
val_steps = len(x_test_img) // BATCH

hist = model.fit(
    train_generator(),
    steps_per_epoch=steps_per_epoch,
    validation_data=val_generator(),
    validation_steps=val_steps,
    epochs=5
)


### Exercise 2 — Use the denoiser

The function below now receives an **already noisy** image. Complete the input construction and test the network.


In [ ]:
def denoise(x_noisy, sigma_value):
    """
    x_noisy: already-noisy images, shape (B, 28, 28, 1)
    sigma_value: float or array with shape (B, 1, 1, 1)
    """
    if np.isscalar(sigma_value):
        sigma = np.full(
            (x_noisy.shape[0], 1, 1, 1),
            float(sigma_value),
            dtype="float32"
        )
    else:
        sigma = sigma_value.astype("float32")

    # TODO: create the two-channel input expected by the CNN.
    # x_in = ...

    return model.predict(x_in, verbose=0)


In [ ]:
# Several images at one noise level.
n = 10
x0 = x_test_img[:n]
sigma_demo = 0.2
sigma = np.full((n, 1, 1, 1), sigma_demo, dtype="float32")

x_noisy = add_noise(x0, sigma)
x_hat   = denoise(x_noisy, sigma_demo)

plt.figure(figsize=(20, 6))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x0[i, ..., 0], cmap="gray")
    ax.set_axis_off()

    ax = plt.subplot(3, n, n + i + 1)
    plt.imshow(x_noisy[i, ..., 0], cmap="gray")
    ax.set_axis_off()

    ax = plt.subplot(3, n, 2 * n + i + 1)
    plt.imshow(x_hat[i, ..., 0], cmap="gray")
    ax.set_axis_off()

plt.suptitle(f"Clean / Noisy (sigma={sigma_demo}) / Denoised")
plt.show()


In [ ]:
# One image at several noise levels.
idx = 0
x0 = x_test_img[idx:idx + 1]
sigmas = np.array([0.00, 0.05, 0.10, 0.20, 0.35, 0.50], dtype="float32")

imgs, titles = [], []
for s in sigmas:
    sigma = np.full((1, 1, 1, 1), s, dtype="float32")
    x_noisy = add_noise(x0, sigma)
    x_hat   = denoise(x_noisy, sigma)
    imgs += [x_noisy[0, ..., 0], x_hat[0, ..., 0]]
    titles += [f"Noisy sigma={s:.2f}", f"Denoised sigma={s:.2f}"]

plt.figure(figsize=(12, 2 * len(sigmas)))
for i, (im, title) in enumerate(zip(imgs, titles), 1):
    ax = plt.subplot(len(sigmas), 2, i)
    plt.imshow(im, cmap="gray")
    ax.set_axis_off()
    ax.set_title(title)
plt.tight_layout()
plt.show()


In [ ]:
def eval_curve(sigmas_eval, N=500):
    idx = np.random.choice(x_test_img.shape[0], size=N, replace=False)
    X = x_test_img[idx]
    mse_list = []

    for s in sigmas_eval:
        sigma = np.full((N, 1, 1, 1), s, dtype="float32")
        X_noisy = add_noise(X, sigma)
        X_hat   = denoise(X_noisy, sigma)
        mse_list.append(np.mean((X_hat - X) ** 2))

    return np.array(mse_list)

sigmas_eval = np.linspace(0.0, 0.8, 9, dtype="float32")
mse_v = eval_curve(sigmas_eval, N=500)

plt.figure(figsize=(8, 3))
plt.plot(sigmas_eval, mse_v, marker="o")
plt.xlabel("Noise level sigma")
plt.ylabel("Denoising MSE")
plt.title("Denoising error vs noise level")
plt.tight_layout()
plt.show()


## Part 2 — From denoising to diffusion

The previous network could denoise several values of $\sigma$. A diffusion model uses the same idea, but the noise level is controlled by a **time step** $t$.

We use $t=0,\ldots,T-1$ and create a noisy image directly with

$$
x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon.
$$

Here $\bar\alpha_t$ is simply a decreasing schedule:

- at the beginning, $\bar\alpha_t \approx 1$, so $x_t$ is almost the clean image;
- at the end, $\bar\alpha_t \approx 0$, so $x_t$ is almost pure Gaussian noise.

We do **not** need to simulate all previous time steps to generate a training example: the formula above gives $x_t$ directly.

The CNN will receive $x_t$ and the time $t$, and will predict the clean image $x_0$. This is called an **$x_0$-prediction** model. Other diffusion models may instead predict the noise, but $x_0$-prediction is easier to visualize here.

### Time information

Rather than giving the CNN one scalar $t$, we give it several sine/cosine channels. For $k=0,\ldots,K-1$, we use

$$
\sin\!\left(2\pi\,2^k\frac{t}{T-1}\right),
\qquad
\cos\!\left(2\pi\,2^k\frac{t}{T-1}\right).
$$

Complete the time embedding below.


In [ ]:
def to_symmetric(x):
    """Map [0, 1] to [-1, 1]."""
    return x * 2.0 - 1.0

def from_symmetric(x):
    """Map [-1, 1] back to [0, 1]."""
    return np.clip((x + 1.0) * 0.5, 0.0, 1.0)


In [ ]:
# Cosine schedule for alpha_bar.
T = 200
u = (np.arange(T) + 0.5) / T
s = 0.008
f = lambda z: np.cos((np.pi / 2) * (z + s) / (1 + s)) ** 2
alpha_bar = (f(u) / f(0)).astype("float32")

plt.figure(figsize=(7, 3))
plt.plot(np.arange(T), alpha_bar)
plt.xlabel("t")
plt.ylabel("alpha_bar[t]")
plt.title("Noise schedule")
plt.tight_layout()
plt.show()

def q_sample(x0_sym, t_idx, eps):
    """Sample x_t directly from x_0."""
    a = np.sqrt(alpha_bar[t_idx]).reshape(-1, 1, 1, 1).astype("float32")
    b = np.sqrt(1.0 - alpha_bar[t_idx]).reshape(-1, 1, 1, 1).astype("float32")
    return a * x0_sym + b * eps

K = 8

def t_embed_map(t_idx, K=K):
    t_normalized = (t_idx.astype("float32") / (T - 1)).reshape(-1, 1, 1, 1)

    # TODO: build frequencies [1, 2, 4, ...].
    # freqs = ...
    # angles = ...
    # sin_features = ...
    # cos_features = ...
    # emb = ...

    return np.tile(emb, (1, 28, 28, 1)).astype("float32")

def make_input(x_t, t_idx):
    return np.concatenate([x_t, t_embed_map(t_idx)], axis=-1).astype("float32")


### Diffusion network

We use a small encoder-decoder with one skip connection. The input has one image channel and $2K$ time channels.


In [ ]:
from keras import Input

CIN = 1 + 2 * K
inp = Input(shape=(28, 28, CIN))

# Encoder: 28x28 -> 14x14
e1 = layers.Conv2D(64, 3, padding="same", activation="swish")(inp)
e1 = layers.Conv2D(64, 3, padding="same", activation="swish")(e1)

e2 = layers.Conv2D(128, 3, strides=2, padding="same", activation="swish")(e1)
e2 = layers.Conv2D(128, 3, padding="same", activation="swish")(e2)

# Bottleneck
b = layers.Conv2D(128, 3, padding="same", activation="swish")(e2)
b = layers.Conv2D(128, 3, padding="same", activation="swish")(b)

# Decoder: 14x14 -> 28x28
d1 = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="swish")(b)
d1 = layers.Concatenate()([d1, e1])
d1 = layers.Conv2D(64, 3, padding="same", activation="swish")(d1)
d1 = layers.Conv2D(64, 3, padding="same", activation="swish")(d1)

# Predict x_0 in [-1, 1].
out = layers.Conv2D(1, 3, padding="same", activation=None)(d1)

model_diff = models.Model(inp, out)
model_diff.compile(optimizer=optimizers.Adam(1e-3), loss="mse")
model_diff.summary()


In [ ]:
BATCH = 256

def diffusion_train_generator():
    N = x_train_img.shape[0]
    while True:
        idx = np.random.randint(0, N, size=(BATCH,))
        x0 = x_train_img[idx]
        x0_sym = to_symmetric(x0)

        t_idx = np.random.randint(0, T, size=(BATCH,), dtype=np.int64)
        eps = np.random.randn(*x0.shape).astype("float32")

        x_t = q_sample(x0_sym, t_idx, eps)
        x_in = make_input(x_t, t_idx)

        yield x_in, x0_sym


### Train the diffusion denoiser

The network now sees examples from all noise levels. On a slow machine you can reduce the number of epochs, but sample quality may decrease.


In [ ]:
steps_per_epoch = len(x_train_img) // BATCH

hist = model_diff.fit(
    diffusion_train_generator(),
    steps_per_epoch=steps_per_epoch,
    epochs=30
)


## Part 3 — Deterministic reverse process

After training, we want to start from noise and move backwards.

At time $t$, the network predicts the clean image

$$
\widehat x_0 = f(x_t,t).
$$

Using the forward equation, we can estimate the noise that is present in $x_t$:

$$
\widehat\varepsilon_t
=
\frac{x_t-\sqrt{\bar\alpha_t}\,\widehat x_0}
     {\sqrt{1-\bar\alpha_t}}.
$$

Then we use the **same estimated clean image and the same estimated noise** at the previous time step:

$$
x_{t-1}
=
\sqrt{\bar\alpha_{t-1}}\,\widehat x_0
+
\sqrt{1-\bar\alpha_{t-1}}\,\widehat\varepsilon_t.
$$

For the final step we use $\bar\alpha_{-1}=1$, so the result becomes the predicted clean image. No fresh randomness is added: starting from the same initial noise gives the same output.

### Exercise 3 — Implement one deterministic reverse step


In [ ]:
def deterministic_step(x_t, t):
    """One deterministic reverse step: x_t -> x_{t-1}."""
    n = x_t.shape[0]
    t_arr = np.full((n,), t, dtype=np.int64)
    x_in = make_input(x_t, t_arr)

    ab_t = float(alpha_bar[t])
    ab_prev = float(alpha_bar[t - 1]) if t > 0 else 1.0

    # TODO: predict x_0, estimate the noise, then move to t-1.
    # x0_hat = ...
    # eps_hat = ...
    # x_prev = ...

    return x_prev


def sample_deterministic(n=16):
    # At the last time step, x_t is approximately standard Gaussian noise.
    x = np.random.randn(n, 28, 28, 1).astype("float32")

    for t in reversed(range(T)):
        x = deterministic_step(x, t)

    return from_symmetric(x)

samples = sample_deterministic(n=16)

plt.figure(figsize=(6, 6))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(samples[i, ..., 0], cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_axis_off()
plt.suptitle("Samples from the deterministic reverse process")
plt.tight_layout()
plt.show()


### Visualize forward noise and deterministic reconstruction

For a smooth visualization we keep the same Gaussian noise realization while increasing $t$. The reverse animation then starts from the final noised image and applies the deterministic step repeatedly.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


def forward_noising_frames(x0_img01, n_frames=30, stop_fraction=1.0):
    x0_sym = to_symmetric(x0_img01)
    frames = [x0_img01[0, ..., 0]]

    t_stop = int(stop_fraction * (T - 1))
    t_seq = np.linspace(0, t_stop, n_frames, dtype=int)

    # Keep the same epsilon so the animation changes smoothly with t.
    eps = np.random.randn(*x0_sym.shape).astype("float32")
    x_t = None
    for t in t_seq:
        x_t = q_sample(x0_sym, np.array([t]), eps)
        frames.append(from_symmetric(x_t[0, ..., 0]))

    return frames, int(t_seq[-1]), x_t


def reverse_deterministic_frames(x_t, t_start, n_frames=30):
    frames = [from_symmetric(x_t[0, ..., 0])]
    t_cur = int(t_start)
    steps_per_frame = max(1, math.ceil((t_cur + 1) / n_frames))

    while t_cur >= 0:
        for _ in range(steps_per_frame):
            if t_cur < 0:
                break
            x_t = deterministic_step(x_t, t_cur)
            t_cur -= 1
        frames.append(from_symmetric(x_t[0, ..., 0]))

    return frames

idx = np.random.randint(0, x_test_img.shape[0])
x0_img = x_test_img[idx:idx + 1]

fwd_frames, t_star, x_t_star = forward_noising_frames(x0_img, n_frames=25)
rev_frames = reverse_deterministic_frames(x_t_star, t_star, n_frames=35)
frames = fwd_frames + rev_frames

fig, ax = plt.subplots()
im = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=1)
ax.axis("off")

def update(i):
    im.set_data(frames[i])
    return [im]

ani = FuncAnimation(fig, update, frames=len(frames), interval=120)
plt.close()
HTML(ani.to_jshtml())


## Part 4 — Stochastic reverse process **[evaluated]**

The deterministic sampler uses no fresh randomness during the reverse trajectory. The original **DDPM** reverse process is stochastic: at every step except the last one, it samples a small Gaussian perturbation.

We will not derive the DDPM formula here. Because the forward process is Gaussian, the corresponding reverse update can be written as a Gaussian step. We simply take the following formulas as given.

For this section only, we need

$$
\alpha_t = \frac{\bar\alpha_t}{\bar\alpha_{t-1}},
\qquad
\beta_t = 1-\alpha_t,
$$

with $\bar\alpha_{-1}=1$.

Given the network prediction $\widehat x_0$, the reverse mean is

$$
\text{mean}
=
\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t
+
\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,\widehat x_0,
$$

and the reverse variance is

$$
\text{var}
=
\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t.
$$

We then sample

$$
x_{t-1}=\text{mean}+\sqrt{\text{var}}\,z,
\qquad z\sim\mathcal N(0,I),
$$

except at $t=0$, where we do not add noise.

### Exercise 4 — Implement the stochastic reverse step

Translate the two formulas above directly into code. The variable names are already chosen to match the notation.


In [ ]:
# Local diffusion coefficients needed by DDPM.
alpha = np.empty_like(alpha_bar)
alpha[0] = alpha_bar[0]              # alpha_bar[-1] is defined as 1
alpha[1:] = alpha_bar[1:] / alpha_bar[:-1]
beta = 1.0 - alpha


def stochastic_step(x_t, t):
    """One DDPM reverse step: x_t -> x_{t-1}."""
    n = x_t.shape[0]
    t_arr = np.full((n,), t, dtype=np.int64)
    x_in = make_input(x_t, t_arr)
    x0_hat = model_diff.predict(x_in, verbose=0).astype("float32")

    ab_t = float(alpha_bar[t])
    ab_prev = float(alpha_bar[t - 1]) if t > 0 else 1.0
    a_t = float(alpha[t])
    b_t = float(beta[t])

    # TODO: translate the mean and variance formulas from the markdown cell.
    # coef_xt = ...
    # coef_x0 = ...
    # mean = ...
    # var = ...

    if t > 0:
        z = np.random.randn(*x_t.shape).astype("float32")
        return mean + np.sqrt(var) * z
    else:
        return mean


def sample_stochastic(n=16):
    x = np.random.randn(n, 28, 28, 1).astype("float32")

    for t in reversed(range(T)):
        x = stochastic_step(x, t)

    return from_symmetric(x)


In [ ]:
samples_stochastic = sample_stochastic(n=16)

plt.figure(figsize=(6, 6))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(samples_stochastic[i, ..., 0], cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_axis_off()
plt.suptitle("Samples from the stochastic reverse process")
plt.tight_layout()
plt.show()


### Deterministic vs stochastic

We now start all four reverse trajectories from the **same** noised image:

- the two deterministic runs should coincide;
- the two stochastic runs can differ because fresh Gaussian noise is sampled during the reverse process.


In [ ]:
def reverse_stochastic_frames(x_t, t_start, n_frames=30):
    frames = [from_symmetric(x_t[0, ..., 0])]
    t_cur = int(t_start)
    steps_per_frame = max(1, math.ceil((t_cur + 1) / n_frames))

    while t_cur >= 0:
        for _ in range(steps_per_frame):
            if t_cur < 0:
                break
            x_t = stochastic_step(x_t, t_cur)
            t_cur -= 1
        frames.append(from_symmetric(x_t[0, ..., 0]))

    return frames


def four_run_diffusion_animation(n_fwd=20, n_back=30):
    idx = np.random.randint(0, x_test_img.shape[0])
    x0 = x_test_img[idx:idx + 1]

    # Stop slightly before the very end so some structure remains visible.
    fwd_frames, t_star, x_t_star = forward_noising_frames(
        x0, n_frames=n_fwd, stop_fraction=0.95
    )

    ddim1 = reverse_deterministic_frames(x_t_star.copy(), t_star, n_frames=n_back)
    ddim2 = reverse_deterministic_frames(x_t_star.copy(), t_star, n_frames=n_back)
    ddpm1 = reverse_stochastic_frames(x_t_star.copy(), t_star, n_frames=n_back)
    ddpm2 = reverse_stochastic_frames(x_t_star.copy(), t_star, n_frames=n_back)

    L = max(len(ddim1), len(ddim2), len(ddpm1), len(ddpm2))
    for seq in (ddim1, ddim2, ddpm1, ddpm2):
        while len(seq) < L:
            seq.append(seq[-1])

    frames = []

    # During the forward phase, show the same image in all four quadrants.
    for fr in fwd_frames:
        top = np.concatenate([fr, fr], axis=1)
        bottom = np.concatenate([fr, fr], axis=1)
        frames.append(np.concatenate([top, bottom], axis=0))

    # During the reverse phase, compare the four trajectories.
    for k in range(L):
        top = np.concatenate([ddim1[k], ddim2[k]], axis=1)
        bottom = np.concatenate([ddpm1[k], ddpm2[k]], axis=1)
        frames.append(np.concatenate([top, bottom], axis=0))

    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(frames[0], cmap="gray", vmin=0, vmax=1)
    ax.axis("off")

    ax.text(-0.05, 0.75, "Deterministic", transform=ax.transAxes,
            va="center", ha="right", fontsize=11)
    ax.text(-0.05, 0.25, "Stochastic", transform=ax.transAxes,
            va="center", ha="right", fontsize=11)

    def update(i):
        im.set_data(frames[i])
        return [im]

    ani = FuncAnimation(fig, update, frames=len(frames), interval=120)
    plt.close()
    return HTML(ani.to_jshtml())

four_run_diffusion_animation(n_fwd=20, n_back=30)


## Optional — A larger network

If the generated digits are poor, the limitation may simply be model size or training time. The larger architecture below was used to produce the attached reference animation. It is optional: the goal of the TP is to understand the diffusion procedure, not to obtain state-of-the-art samples.


In [ ]:
# CIN = 1 + 2 * K
# inp = Input(shape=(28, 28, CIN))
#
# # Encoder: 28x28
# e1 = layers.Conv2D(256, 3, padding="same", activation="swish")(inp)
# e1 = layers.Conv2D(256, 3, padding="same", activation="swish")(e1)
#
# # Encoder: 14x14
# e2 = layers.Conv2D(512, 3, strides=2, padding="same", activation="swish")(e1)
# e2 = layers.Conv2D(512, 3, padding="same", activation="swish")(e2)
#
# # Bottleneck
# b = layers.Conv2D(512, 3, padding="same", activation="swish")(e2)
# b = layers.Conv2D(512, 3, padding="same", activation="swish")(b)
#
# # Decoder: 14x14 -> 28x28
# d1 = layers.Conv2DTranspose(256, 3, strides=2, padding="same", activation="swish")(b)
# d1 = layers.Concatenate()([d1, e1])
# d1 = layers.Conv2D(256, 3, padding="same", activation="swish")(d1)
# d1 = layers.Conv2D(256, 3, padding="same", activation="swish")(d1)
#
# out = layers.Conv2D(1, 3, padding="same", activation=None)(d1)
#
# model_diff = models.Model(inp, out)
# model_diff.compile(optimizer=optimizers.Adam(1e-3), loss="mse")
# model_diff.summary()


![Reference diffusion animation](diffusion.gif)
